# MAX-3 · Maximum Model — Single-Fold Test (MRKR)

Trains the maximum configuration on the MRKR fold: CORN ordinal head, full
backbone fine-tuning, 384×384 resolution, soft ordinal targets, source-reliability
weighting, sampling, curriculum, and domain-adversarial training. Results are
written to scope3_max; the original pipeline is untouched. This single fold
confirms whether the maximum stack improves on the baseline before running all
four folds.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np, torch
if 'training_lib_max' in sys.modules: importlib.reload(sys.modules['training_lib_max'])
import training_lib_max as TM
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> Change runtime type -> GPU (A100).')
print('GPU:', torch.cuda.get_device_name(0))
print('Resolution:', TM.MAX_IMG_SIZE, '| Freeze:', TM.MAX_FREEZE_STAGES, '| MRKR trust:', TM.MRKR_TRUST)
manifest = TM.prepare_local_data()
print(f'Manifest: {len(manifest):,} rows')

Mounted at /content/drive
GPU: NVIDIA A100-SXM4-40GB
Resolution: 384 | Freeze: 0 | MRKR trust: 0.4
Copied array in 41s
Loaded array (61558, 224, 224) in 1s
Manifest: 61,558 rows


## Train maximum model on Mendeley fold

In [2]:
mech = {'sampler':True,'noise_loss':True,'curriculum':True,'domain_adv':True,
        'hierarchical':True,'use_quality':True,'soft_alpha':TM.MAX_SOFT_ALPHA,
        'freeze_stages':TM.MAX_FREEZE_STAGES,'grl_lambda_max':0.5}

test_ds = 'mrkr'
tr, va, te = TM.make_splits(manifest, test_ds, seed=0)
print(f'train={len(tr):,}  val={len(va):,}  test={len(te):,}')

res = TM.run_training('max_mrkr_seed0', tr, va, te, mech, seed=0,
                      log_fn=print)

train=18,352  val=3,239  test=39,967
Downloading: "https://download.pytorch.org/models/convnext_large-ea097f82.pth" to /root/.cache/torch/hub/checkpoints/convnext_large-ea097f82.pth


100%|██████████| 755M/755M [00:03<00:00, 227MB/s]


  [max_mrkr_seed0] ep1/18 loss=1.400 tr=0.237 val=0.423 w1=0.591 qwk=0.021 gap=-0.186 (253s)
  [max_mrkr_seed0] ep2/18 loss=1.020 tr=0.402 val=0.525 w1=0.780 qwk=0.557 gap=-0.123 (225s)
  [max_mrkr_seed0] ep3/18 loss=1.158 tr=0.498 val=0.541 w1=0.796 qwk=0.589 gap=-0.042 (226s)
  [max_mrkr_seed0] ep4/18 loss=1.325 tr=0.531 val=0.579 w1=0.831 qwk=0.670 gap=-0.049 (225s)
  [max_mrkr_seed0] ep5/18 loss=1.287 tr=0.557 val=0.592 w1=0.849 qwk=0.688 gap=-0.035 (226s)
  [max_mrkr_seed0] ep6/18 loss=2.470 tr=0.449 val=0.420 w1=0.585 qwk=0.000 gap=+0.029 (226s)
  [max_mrkr_seed0] ep7/18 loss=2.071 tr=0.441 val=0.589 w1=0.853 qwk=0.704 gap=-0.148 (225s)
  [max_mrkr_seed0] ep8/18 loss=1.850 tr=0.566 val=0.592 w1=0.845 qwk=0.684 gap=-0.027 (227s)
  [max_mrkr_seed0] ep9/18 loss=1.921 tr=0.567 val=0.613 w1=0.862 qwk=0.725 gap=-0.046 (226s)
  [max_mrkr_seed0] ep10/18 loss=1.885 tr=0.590 val=0.607 w1=0.856 qwk=0.707 gap=-0.017 (226s)
  [max_mrkr_seed0] ep11/18 loss=1.818 tr=0.595 val=0.620 w1=0.872 qwk

## Compare to baseline (original H, Mendeley fold)

In [3]:
import json, glob
def load(p):
    try: return json.load(open(str(p)))
    except Exception: return None

h = load(config.RESULTS_DIR/'fold2_test_mrkr_seed0.json')
h_w1 = np.nan
zf = sorted(glob.glob(str(config.RESULTS_DIR/'fold2_test_mrkr_seed*_preds.npz')))
if zf:
    z=np.load(zf[0]); h_w1=float((np.abs(z['y_true']-z['y_pred'])<=1).mean())

print('MRKR fold — baseline H vs MAX (seed 0):')
print(f"{'metric':12s}{'H (224)':>12s}{'MAX (384)':>12s}")
if h:
    print(f"{'exact acc':12s}{h['external_acc5']:>12.3f}{res['external_acc5']:>12.3f}")
    print(f"{'within-1':12s}{h_w1:>12.3f}{res['external_within1']:>12.3f}")
    print(f"{'QWK':12s}{h['external_qwk']:>12.3f}{res['external_qwk']:>12.3f}")
    print(f"{'gap (pp)':12s}{h['gap']*100:>12.1f}{res['gap']*100:>12.1f}")
    print()
    print(f"Delta exact: {res['external_acc5']-h['external_acc5']:+.3f}")
    print(f"Delta within-1: {res['external_within1']-h_w1:+.3f}")
else:
    print(f"MAX: exact={res['external_acc5']:.3f} within1={res['external_within1']:.3f} qwk={res['external_qwk']:.3f} gap={res['gap']*100:.1f}pp")

MRKR fold — baseline H vs MAX (seed 0):
metric           H (224)   MAX (384)
exact acc          0.476       0.511
within-1           0.681       0.734
QWK                0.508       0.608
gap (pp)             7.6        12.3

Delta exact: +0.034
Delta within-1: +0.054
